## Spatial Data Science with CityJSON

The purpose of this Notebook is to ***work with*** the product of [osm_LoD1_3DCityModel](https://github.com/AdrianKriger/osm_LoD1_3DCityModel); a previously created CityJSON city model.

<div class="alert alert-block alert-warning"><b>This notebook will:</b>

> **1. allow the user to execute an application of Spatial Data Science**  
>
>> **a)  [population estimation](#Section1a)** _--with a previous census metric population growth rate and projected (future) population are also possible_  **and**  
>> **b)  a measure of [Building Volume per Capita](#Section1b)**
>
> **2. further applications of Spatial Data Science**  
>
>> i calculate percentage **homes and population with direct access to on-site renewable energy infrastructure** *--rooftop photovoltaic panels (PV) and solar water heaters (SWH).*
>>
>> ii. estimate the [**Annual Average Solar** *(photovoltaic)* **Potential**](https://www.worldbank.org/en/topic/energy/publication/solar-photovoltaic-power-potential-by-country), per home.
>
> **3. produce [an interactive visualization](#Section1b)** *- which a user can navigate, query and share* **that**;
> > **a) [colour buildings by type](#Section2a)** *(to easily visualize building stock)* 
>
> **4. propose several [Geography and Sustainable Development Education *conversation starters*](#Section3) for Secondary and Tertiary level students**
</div>

<div class="alert alert-block alert-danger"><b>Please Note:</b>

***The [village](https://github.com/AdrianKriger/geo3D/tree/main/village)*** processing option is meant for areas with no more than for **2 500 buildings**.</div>

In [1]:
#- load the magic

%matplotlib inline
import os
import sys
from pathlib import Path
import requests

import numpy as np
import pandas as pd
import shapely
from shapely.geometry import Polygon, shape, mapping
import json
import geojson

from cjio import cityjson

import matplotlib.pyplot as plt
#import pydeck as pdk

In [2]:
#- get current working directory (notebook location)
current_dir = os.getcwd()

#- go one level up
parent_dir = os.path.dirname(current_dir)
# Add parent directory to sys.path if needed
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

#- import
import city3D

In [3]:
import warnings
warnings.filterwarnings('ignore')

**The area under investigation is [Mamre, Cape Town. South Africa](https://en.wikipedia.org/wiki/Mamre,_South_Africa).**

In [4]:
#- change to harvest the appropriate CityJSON

#jparams = json.load(open('../param/uEstate_param.json'))
#jparams = json.load(open('../param/cput_param.json'))
#jparams = json.load(open('../param/saao_param.json'))
jparams = json.load(open('../param/mamre_param.json'))
#jparams = json.load(open('../param/sRiver_param.json'))

In [5]:
#cm = cityjson.load(path=jparams['cjsn_solid']) #-- citjsnClean_rural3D.json in the result folder

# With the new 0.10.x syntax:
with open(jparams['cjsn_out'], 'r') as f:
    cm = cityjson.reader(f)

In [6]:
print(cm)

CityJSON version = 2.0
EPSG = 32734
bbox = [ 263698.088 6287743.433 138.270 266820.587 6290344.540 270.690 ]
=== CityObjects ===
|-- TINRelief (1)
|-- Building (2340)
materials = False
textures = False


In [7]:
#- no longer supported
#df = cityjson.to_dataframe(cm)

# Extract data directly from the dictionary we built
data = []
for obj_id, obj_data in cm.j['CityObjects'].items():
    # We only want Buildings for this GDF, skipping the terrain (TINRelief)
    if obj_data['type'] == 'Building':
        row = {'id': obj_id}
        row.update(obj_data.get('attributes', {}))
        
        # In your building loop, the footprint was 'row.geometry'
        # If you need to re-harvest it from the CityJSON structure:
        # For LoD1, we usually grab the 'GroundSurface' or the first floor
        # But since you already have gdf_blds, it's easier to use that!
        data.append(row)

df = pd.DataFrame(data)

# Harvest the CRS from the cm_obj metadata we set earlier
# Using the new object-oriented method
crs_string = cm.j['metadata']['referenceSystem'] 
# - we split by the last slash
final_crs = crs_string.split('/')[-1]

In [8]:
#df = cm.to_dataframe()
#- remove the first feature: the terrain
#df = df[1:]    

#- harvest the crs
theinfo = cm.get_info()
crs = theinfo[1]

# account for holes
def coords_to_polygon(rings):
    if not rings or len(rings) == 0:
        return None
    outer = rings[0]
    # Only pass holes if they actually exist in the list
    holes = rings[1:] if len(rings) > 1 else []
    return Polygon(shell=outer, holes=holes)

# Convert JSON string to Python list
#df['footprint_coords_list'] = df['footprint'].apply(json.loads)
# Only apply json.loads if the entry is actually a string
df['footprint_coords_list'] = df['footprint'].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)

# create home-baked gdf
gdf = city3D.GeoDataFrameLite(df)
gdf['geometry'] = gdf['footprint_coords_list'].apply(coords_to_polygon)
gdf.crs = crs[7:]
# Drop columns inplace
gdf.drop(columns=['footprint', 'footprint_coords_list'], inplace=True)
#gdf.head(2)

## 1. Spatial Data Science

<div class="alert alert-block alert-warning"><b>We start with basic spatial analysis</b>  
    
     
- We'll [estimate the population](#Section1a), within our area of interest, and then  
- calculate the [Building Volume Per Capita (BVPC)](#Section1b).
</div>

While estimating population is well documented; recent investigations to **understand overcrowding** have led to newer measurements.  

The most noteable of these is **Building Volume Per Capita (BVPC)** [(Ghosh, T; et al. 2020)](https://www.researchgate.net/publication/343185735_Building_Volume_Per_Capita_BVPC_A_Spatially_Explicit_Measure_of_Inequality_Relevant_to_the_SDGs). BVPC is the cubic meters of building per person. **BVPC tells us how much space one person has per residential living unit** (a house / apartment / etc.). It is ***a proxy measure of economic inequality and a direct measure of housing inequality***.

BVPC builds on the work of [(Reddy, A and Leslie, T.F., 2013)](https://www.tandfonline.com/doi/abs/10.1080/02723638.2015.1060696?journalCode=rurb20) and attempts to integrate with several **[Sustainable Development Goals](https://sdgs.un.org/goals)** (most noteably: **[SDG 11: Developing sustainable cities and communities](https://sdgs.un.org/goals/goal11)**) and captures the average ***'living space'*** each person has in their home.

<div class="alert alert-block alert-info"><b>These analysis expect the user to have some basic knowledge about the environment under inquiry / investigation</b> </div>

In [9]:
gdf.head(2)

,id,osm_id,building,building:levels,building_height,plus_code,ground_height,roof_height,address,amenity,operator,residential,min_height,bottom_roof_height,building:use,geometry
0,osm_328118446,328118446,yes,1,4.1,4FRWFFVC+P79,179.18,183.28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((265041.635 6289775.059, 265050.531 6..."
1,osm_328118447,328118447,church,2,6.9,4FRWFFVC+H9Q,178.30,185.20,Moravian Church South Africa Kerk Street Mamre...,place_of_worship,NaN,NaN,NaN,NaN,NaN,"POLYGON ((265070.977 6289761.325, 265072.315 6..."


In [10]:
#gdf.plot()
# have a look at the building type and amenities available
gdf['building'].unique()

array(['yes', 'church', 'house', 'cabin', 'public', 'civic', 'office',
       'retail', 'clinic', 'school', 'garage', 'greenhouse', 'roof',
       'kindergarten', 'construction', 'clubhouse', 'guest_house',
       'service', 'detached', 'shed'], dtype=object)

<a id='Section1a'></a>

### 1. a) Estimate Population

<div class="alert alert-block alert-success"><b></b> 
    
_(with population growth rate and population projection possible too)_ </div>

In [11]:
#- some data wrangling to replace 'bld:residential' to 'bld:student' if 'residential:student'
gdf_pop = gdf.copy()

if 'residential' in gdf_pop.columns:
    df_res = gdf_pop[gdf_pop['residential'] == 'student']
    #df_res = df2[df2['building:use'] != None]
    df_res = df_res[~df_res['residential'].isna()]
    gdf_pop.loc[df_res.index, 'building'] = df_res['residential'] 

#- some more data wrangling
with pd.option_context("future.no_silent_downcasting", True):
    if 'building:flats' in gdf_pop.columns: 
        gdf_pop['building:flats'] = pd.to_numeric(gdf_pop['building:flats'].fillna(0).infer_objects(copy=False))
    if 'building:units' in gdf_pop.columns:    
        gdf_pop['building:units'] = pd.to_numeric(gdf_pop['building:units'].fillna(0).infer_objects(copy=False))
    if 'beds' in gdf_pop.columns:   
        gdf_pop['beds'] = pd.to_numeric(gdf_pop['beds'].fillna(0).infer_objects(copy=False))
    if 'rooms' in gdf_pop.columns:   
        gdf_pop['rooms'] = pd.to_numeric(gdf_pop['rooms'].fillna(0).infer_objects(copy=False))

gdf_pop["building:levels"] = pd.to_numeric(gdf_pop["building:levels"])

gdf_pop.head(2)

,id,osm_id,building,building:levels,building_height,plus_code,ground_height,roof_height,address,amenity,operator,residential,min_height,bottom_roof_height,building:use,geometry
0,osm_328118446,328118446,yes,1,4.1,4FRWFFVC+P79,179.18,183.28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((265041.635 6289775.059, 265050.531 6..."
1,osm_328118447,328118447,church,2,6.9,4FRWFFVC+H9Q,178.30,185.20,Moravian Church South Africa Kerk Street Mamre...,place_of_worship,NaN,NaN,NaN,NaN,NaN,"POLYGON ((265070.977 6289761.325, 265072.315 6..."


In [12]:
gdf_pop['building'].value_counts()

building
house           1642
cabin            382
yes              157
garage           112
retail            11
civic              7
roof               7
school             5
church             3
public             2
construction       2
detached           2
office             1
clinic             1
greenhouse         1
kindergarten       1
clubhouse          1
guest_house        1
service            1
shed               1
Name: count, dtype: int64

**This area** (Mamre) **is peri-urban with single level housing units. To estimate population is thus pretty straight forward.**

<div class="alert alert-block alert-info"><b>We start with local knowledge.</b></div>

**On average there are roughly `6` people per `building:house` in this area.**  

An ***informal*** structure ([shack](https://en.wikipedia.org/wiki/Shack)) is tagged [building:cabin](https://wiki.openstreetmap.org/wiki/Tag:building%3Dcabin) and houses `4` people.

<div class="alert alert-block alert-danger"><b>Your Participation! </b>
    

We will execute the calculation programmatically. **Fill in the relevant variables in the _`cell`_ below** </div>

In [13]:
#- average number of residents per formal house
f_house = 6
#- average number of residents per informal structure
inf_structure = 4

<div class="alert alert-block alert-warning"><b></b>  
    
**Furthermore:**  
    - **[social housing](https://en.wikipedia.org/wiki/Public_housing)** is tagged `building:residential` with the number of occupants iether *the number of informal structure occupants* or `building:flats * inf_structure`  
    - A `social_facility` (carehome, shelter, etc.) harvests the `beds` *'key:value'* pair.  
    - `building:apartment` harvests the `building:flats` *'key:value'* pair *(the number of units)* to calculate `*3` people per apartment.  
    - ***Student accomodation***:  
>    - University owed: is tagged `building:dormitory` with `residential:university` and harvests the `beds` *'key:value'* pair.
>    - Private for-profit: is tagged `building:residential` or `:dormitory` with `residential:student` and then harvests the `building:flats` or `:rooms` *'key:value'* pair *(the number of units)* to calculate `*1` people per apartment; if `level: > 1` else `*3` people in a house share.
    
**The tagging scheme and numbers is based on *how your community is mapped* and local knowledge**
</div>

In [14]:
c = gdf_pop.columns

def pop(row):
    #- formal house
    if row['building'] == 'house' or row['building'] == 'semidetached_house':
        return f_house
    if row['building'] == 'terrace' and 'building:units' in c or row['building'] == 'terraced' and 'building:units' in c:
        return row['building:units'] * f_house

    #- informal structure (shack)
    if row['building'] == 'cabin':
        return inf_structure

    #- in this case social housing
    if row['building'] == 'residential' and 'social_facility' in c and row['social_facility'] is np.nan:
        if row['building:levels'] > 1:
            if 'rooms' in c and row['rooms'] != 0:
                return row['rooms']
            if 'building:flats' in c and row['building:flats'] != 0:
                return row['building:flats'] * inf_structure
        else:
            return inf_structure

    #-- social facility [shelter / carehome]
    if row['building'] == 'residential' and 'social_facility' in c and row['social_facility'] is not np.nan:
        if row['building:levels'] > 1:
            if 'building:units' in c and row['building:units'] != 0:
                return row['building:units'] 
            if 'beds' in c and row['beds'] != 0:
                return row['beds']
        else:
            return inf_structure

    #- formal apartment
    if row['building'] == 'apartments':
        if 'rooms' in c and row['rooms'] != 0:
            return row['rooms']
        else:
            return row['building:flats'] * 3
        
    #- private student residence 
    if row['building'] == 'student':
        if row['building:levels'] > 1:
            if 'rooms' in c and row['rooms'] != 0:
                return row['rooms']
            else:
                return row['building:flats']
        else:
            return 3
            
    # university owned student residence
    if row['building'] == 'dormitory' and row['residential'] == 'university':
        if row['building:levels'] > 1:
            if 'rooms' in c and row['rooms'] != 0:
                return row['rooms']
            if 'beds' in c and row['beds'] != 0:
                return row['beds']
        else:
            return 3

gdf_pop['pop'] = gdf_pop.apply(lambda x: pop(x), axis=1)

est_pop = gdf_pop['pop'].sum()
print('The estimated population is:', est_pop)

The estimated population is: 11380.0


**The official [STATSSA 2011 census figure](https://www.statssa.gov.za/?page_id=4286&id=291), for this community, is 9048.**

We can calculate the annual population growth rate using the formula for **[Annual population growth](https://databank.worldbank.org/metadataglossary/health-nutrition-and-population-statistics/series/SP.POP.GROW):**

$$r = \frac{\ln{[\frac{End Population}{Start Population}}]}{n} * 100 = \frac{\ln{[\frac{11 120^{*}}{9048}}]}{12} * 100   = 1.43\%$$
<br>
<sup>* <sub>***Notice!*** The estimated population (11176) is **NOT** the number in the formula (11 120). This community is frequently updated on OpenStreetMap and variations are common.</sub> 

<div class="alert alert-block alert-danger"><b>Your Participation! </b>
    

It is possible to execute the calculation programmatically. **Fill in the relevant variables in the _`cell`_ below** </div>

In [15]:
#- previous population
start_population = 9048                                #- uEstate estimate: 987 

#- period in years from the previous census
years = 12

In [16]:
#-execute
r = (np.log(est_pop/start_population)/years) * 100
print('population growth rate of approximately:', round(r, 2), '%')

population growth rate of approximately: 1.91 %


To conclude; we can project into the future with a very basic formula to estimate the population _x_-years from now:  

$$p  = P_o * (1 + r)^{t} = p = 10736 * (1 + 0.0143)^{10}  = 12 368$$

<div class="alert alert-block alert-danger"><b>Your Participation! </b>
    

It is possible to execute the calculation programmatically. **Fill in the variables in the _`cell`_ below** </div>

In [17]:
#- period in years from now
years = 10

In [18]:
#p = est_pop * (1 + (r/100))**years

#print('estimated population', years ,'years from now:', int(p))

#- account for non-residential areas without failure
#- helper function
def safe_population_estimate(est_pop, r, years):
    try:
        p = est_pop * (1 + (r / 100))**years
        return int(p)
    except Exception as e:
        print(f"Population estimate failed: {e}")
        return None  # keeps notebook running

#- execute function
p = safe_population_estimate(est_pop, r, years)

#- shows error and moves on
if p is not None:
    print(f"estimated population {years} years from now: {p}")

estimated population 10 years from now: 13751


<a id='Section1b'></a>

### 1. b) Building Volume Per Capita (BVPC)
<div class="alert alert-block alert-success"><b></b>  
BVPC = total population of a community divided by sum of building volume</div>

In [19]:
#gdf_pop.head(3)

In [20]:
#gdf_pop['area'] = gdf_pop['geometry'].area#\.map(lambda p: p.area)
gdf_pop['area'] = gdf_pop['geometry'].apply(lambda geom: geom.area if geom else 0)
gdf_pop['volume'] = gdf_pop['area'] * gdf_pop['building_height']

#- remove the volume of the ground floor (unoccupied) when building:levels > 7 [this is an arbitrary number based on local knowledge]
#- typically the space is reserved for some other function: retail, etc. 
gdf_pop['volume'] = [
    (row['volume'] - row['area'] * 2.8) if (
        'social_facility' in row and row['social_facility'] is np.nan and row['building:levels'] > 7 and
        row['building'] in ['residential', 'apartments', 'student']
    ) else row['volume']
    for _, row in gdf_pop.iterrows()
]

#gdf_pop['bvpc'] =  gdf_pop['volume'] / gdf_pop['pop']
gdf_pop['bvpc'] = np.where(
    gdf_pop['pop'] > 0,
    gdf_pop['volume'] / gdf_pop['pop'],
    np.nan
)

gdf_pop.tail(2)

,id,osm_id,building,building:levels,building_height,plus_code,ground_height,roof_height,address,amenity,operator,residential,min_height,bottom_roof_height,building:use,geometry,pop,area,volume,bvpc
2338,osm_12289266,12289266,house,1,4.1,4FRWFFMF+W7J,185.38,189.48,22 Clarkeson Street Mamre 7347 Cape Town,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((265302.872 6288753.93, 265286.967 62...",6.0,344.676515,1413.173712,235.528952
2339,osm_12357148,12357148,house,1,4.1,4FRWFFPJ+P7W,192.97,197.07,2 Tol Street Mamre 7347 Cape Town,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((265989.649 6288995.735, 265988.344 6...",6.0,329.185855,1349.662006,224.943668


In [21]:
print(gdf_pop['bvpc'].describe())

count    2024.000000
mean       74.953154
std        49.890327
min         7.418187
25%        32.708953
50%        65.310400
75%       101.574829
max       399.733661
Name: bvpc, dtype: float64


In [22]:
bvpc = round(gdf_pop['volume'].sum() / est_pop, 3)

print('Building Volume Per Capita (BVPC):', bvpc)

Building Volume Per Capita (BVPC): 89.543


<div class="alert alert-block alert-info"><b></b>

**This BVPC value is for all buildings with a `population > 0`. Buildings people live in** *(homes)*. 

And we can seperate `building:house` from `building:cabin` and `building:residential` to undertand the differences between ***formal and informal*** housing in this area.
    
**We want to understand the living space *(the cubic-meter BVPC value)* each person has in thier home**
</div>

In [23]:
formal = gdf_pop[gdf_pop["building"].isin(['house', 'semidetached_house', 'terrace', 'terraced', 'apartments'])].copy()
f_pop = formal['pop'].sum()
#f_area = formal['area'].mean()

informal = gdf_pop[gdf_pop["building"].isin(['residential', 'cabin'])].copy()
inf_pop = informal['pop'].sum()
#inf_area = formal['area'].mean()

#- student
stu = gdf_pop[gdf_pop["building"].isin(['student', 'dormitory'])].copy()
stu_pop = stu['pop'].sum()

bvpc_formal = round(formal['volume'].sum() / formal['pop'].sum()if formal['pop'].sum() != 0 else 0, 3)
bvpc_informal = round(informal['volume'].sum() / informal['pop'].sum() if informal['pop'].sum() != 0 else 0, 3)
bvpc_stu = round(stu['volume'].sum() / stu['pop'].sum() if stu['pop'].sum() != 0 else 0, 3)

print('FORMAL: Population: ', f_pop, ' with Building Volume Per Capita (BVPC):', bvpc_formal)
print('')
print('STUDENT RESIDENCE: Population: ', stu_pop, ' with Building Volume Per Capita (BVPC):', bvpc_stu)
print('')
print('INFORMAL: Population: ', inf_pop, ' with Building Volume Per Capita (BVPC)', bvpc_informal)

FORMAL: Population:  9852.0  with Building Volume Per Capita (BVPC): 83.854

STUDENT RESIDENCE: Population:  0.0  with Building Volume Per Capita (BVPC): 0

INFORMAL: Population:  1528.0  with Building Volume Per Capita (BVPC) 36.694


<div class="alert alert-block alert-danger"><b>Warning: </b>
    

These are LoD1 3D City Models and works well in these types of areas.  
LoD2 would offer a more representative BVpC [(Ghosh, T; et al. 2020)](https://www.researchgate.net/publication/343185735_Building_Volume_Per_Capita_BVPC_A_Spatially_Explicit_Measure_of_Inequality_Relevant_to_the_SDGs) value; when the complexity of the built environment increases.  

Think about a `house` with living space in the roof structure, so called *'attic living'*, or an `apartment` / `residential` building with different levels, loft apartments and/or units in the turrets of a `building`. 

***consider***: this area seperates [building:cabin](https://wiki.openstreetmap.org/wiki/Tag:building%3Dcabin) from `building:residential` to more precisely represent informal structures without typical roof trussess but account for [social housing](https://en.wikipedia.org/wiki/Public_housing) that does</div>

## 2. Further examples of Spatial Data Science *(renewable energy)*:

<div class="alert alert-block alert-warning"><b></b>

**Let's attempt to understand the % of homes and population served with renewable energy.**
</div>
    
[**SDG**](https://sdgs.un.org/goals) indicators are typically calculated at **region and national scales**.  
Here, because we are working with highly detailed, local data, we can explore what a [**Tier 3 local indicator**](https://unstats.un.org/sdgs/metadata/) might look like at a ***neighbourhood level***.
<br>

In this section 3. we evaluate [**SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all**](https://sdgs.un.org/goals/goal7) at a community level and estimate the **proportion of residential units and population** that have **direct access to on-site renewable energy infrastructure** *--rooftop photovoltaic panels (PV) and solar water heaters (SWH)*.

- Percentage of **households** served by rooftop renewable energy  
- Percentage of the **population** served by rooftop renewable energy
</div>

In [25]:
#- harvest rooftop solar

query = """
     [out:json][timeout:180];
    // --when areas have duplicate names given the world has a limited amount of uniquely named places
    area[name='{0}'] ->.b;
    // -- target area ~ can be way or relation
    wr(area.b)[name='{1}'];
    map_to_area -> .a;
    (
        way["power"="generator"]["generator:source"="solar"](area.a);                // Catches simple generators
       //  way["power"="solar_photovoltaic_panel"](area.a);                          // Catches the alternate tag
    );
    out geom;
    """.format(jparams['LargeArea'], jparams['FocusArea'])

#- execute function from city3D, harvest generator solar and return GeoDataFrameLite | home-baked gdf
sol = city3D.overpass_to_gdf(query)
#sol = sol.to_crs(epsg)
sol = sol.to_crs(crs[7:])


if len(sol) < 0:
    print("\033[0m No rooftop solar are mapped in", jparams['FocusArea'])

In [26]:
#- look
sol.head(2)

,generator:method,generator:output:hot_water,generator:source,generator:type,location,power,start_date,area,generator:output:electricity,geometry,osm_id,osm_type
0,thermal,yes,solar,solar_thermal_collector,roof,generator,NaN,NaN,NaN,"POLYGON ((266074.7976815852 6288576.002370844,...",1095737675,way
1,thermal,yes,solar,solar_thermal_collector,roof,generator,NaN,NaN,NaN,POLYGON ((266194.47713799647 6288764.539267579...,1095739760,way


In [27]:
#- the number of renewable in AREA
sol['generator:method'].value_counts()

generator:method
thermal         57
photovoltaic     1
Name: count, dtype: int64

In [28]:
# join (link) rooftop renewable energy to the appropriate bld
def buildings_with_solar(gdf_buildings, gdf_solar):
    # Prepare output arrays
    solar_ids_per_building = [[] for _ in range(len(gdf_buildings["geometry"]))]
    solar_types_per_building = [[] for _ in range(len(gdf_buildings))]
    
    for i, b_geom in enumerate(gdf_buildings["geometry"]):
        for j, s_geom in enumerate(gdf_solar["geometry"]):
            #if b_geom.intersects(s_geom):
            if b_geom.contains(s_geom):
                solar_ids_per_building[i].append(gdf_solar["osm_id"].iloc[j])
                solar_types_per_building[i].append(gdf_solar["generator:method"].iloc[j])

    #- keep only unique values
    #unique_methods_per_building = [list(set(lst)) for lst in solar_types_per_building]
    
    gdf_buildings["solar_ids"] = solar_ids_per_building
    gdf_buildings["generator:method"] = solar_types_per_building #unique_methods_per_building
    gdf_buildings["has_solar"] = [len(lst) > 0 for lst in solar_ids_per_building]
    gdf_buildings["solar_ids"] = solar_ids_per_building
    
    return gdf_buildings

blds = buildings_with_solar(gdf_pop, sol)
blds.head(2)

,id,osm_id,building,building:levels,building_height,plus_code,ground_height,roof_height,address,amenity,...,bottom_roof_height,building:use,geometry,pop,area,volume,bvpc,solar_ids,generator:method,has_solar
0,osm_328118446,328118446,yes,1,4.1,4FRWFFVC+P79,179.18,183.28,NaN,NaN,...,NaN,NaN,"POLYGON ((265041.635 6289775.059, 265050.531 6...",NaN,268.403360,1100.453776,NaN,[],[],False
1,osm_328118447,328118447,church,2,6.9,4FRWFFVC+H9Q,178.30,185.20,Moravian Church South Africa Kerk Street Mamre...,place_of_worship,...,NaN,NaN,"POLYGON ((265070.977 6289761.325, 265072.315 6...",NaN,371.164869,2561.037596,NaN,[],[],False


In [29]:
#--we only want buildings people live in (homes). building=house or =apartment or =residential, etc.
blds = blds[blds["building"].isin(['house', 'semidetached_house', 'terrace', 'terraced', 'apartments', 'residential', 'dormitory', 'cabin', 'garage'])].copy()

<div class="alert alert-block alert-success"><b>Percentage of households served by rooftop renewable energy</b></div>

$$ \text{\% homes with renewable energy} = \frac{\text{Number of dwellings with mapped solar PV or SWH}}{\text{Total number of dwellings}} \times 100 $$

In [30]:
#- harvest columns
with_solar = sum(blds["has_solar"])
pop = est_pop #gdf["pop"]
total_homes = len(blds)

solHms = round(with_solar / total_homes * 100, 2)

print('\033[1m Percentage homes, \033[0m in', jparams['FocusArea'],', with rooftop photovoltaic panels (PV) and solar water heaters (SWH):', solHms)

 Percentage homes,  in Mamre , with rooftop photovoltaic panels (PV) and solar water heaters (SWH): 2.62


<div style="text-align: left;"> 
<small> <b>NB:</b> this number includes the <a href="https://wiki.openstreetmap.org/wiki/Tag:building%3Dgarage">OpenStreetMap <code>building=garage</code></a> building type. Go to <code>Cell &plusmn;27</code> (above) to exclude this building type from the estimate.</small>
</div>
<br>
<div class="alert alert-block alert-success"><b>Percentage of population served by rooftop renewable energy</b></div>

$$ \text{\% population with renewable energy} = \frac{\text{Number of residents with mapped solar PV and SWH}}{\text{Estimated population}} \times 100 $$

In [31]:
#pop_total = gdf_pop["pop"].sum()
pop_total = blds["pop"].sum()
#pop_solar = gdf_pop["pop"][blds["has_solar"]].sum()
pop_solar = blds["pop"][blds["has_solar"]].sum()


solPop = round(pop_solar / pop_total * 100, 2)

print('\033[1m Percentage population, \033[0m in', jparams['FocusArea'],', with rooftop photovoltaic panels (PV) and solar water heaters (SWH):', solPop)

 Percentage population,  in Mamre , with rooftop photovoltaic panels (PV) and solar water heaters (SWH): 2.95


In [32]:
# number of solar renewable on HOMES
blds['generator:method'].explode().value_counts()

generator:method
thermal         55
photovoltaic     1
Name: count, dtype: int64

### 2. ii) Solar potential (MWh)

<div class="alert alert-block alert-warning"><b></b>

In this section, we attempt to understand how much ***'fuel'*** a rooftop can get from the sun.
</div>

We are calling the [NASA POWER API (Prediction Of Worldwide Energy Resources)](https://power.larc.nasa.gov/docs/services/api/temporal/monthly/). This is a global dataset that uses [NASA satellite observations and weather models](https://power.larc.nasa.gov) to tell us exactly how much solar radiation hits a specific location on Earth.

What we are requesting:

> **Parameter:** `ALLSKY_SFC_SW_DWN` (Global Horizontal Irradiance): a 30-year historical average of solar radiation. This ensures our communities solar potential is based on ***long-term climate trends*** rather than a single year of weather.
>
> **Source:** [NASA POWER Climatology API](https://power.larc.nasa.gov/docs/services/api/temporal/climatology/)
>   
> **Goal:**
> To calculate the Annual Total GHI (kWh/m2/year). This value tells us the cumulative **'solar pressure'** hitting our rooftops over an entire year, which we then use to calculate **how many Megawatt-hours (MWh) of clean electricity our neighborhood can generate**.

In [33]:
def get_ghi_data(lat, lon):#, year="2020"):
    #url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    #url = "https://power.larc.nasa.gov/api/temporal/monthly/point"
    url = "https://power.larc.nasa.gov/api/temporal/climatology/point"
    params = {
        "parameters": "ALLSKY_SFC_SW_DWN", # This is NASA's GHI code
        "community": "RE",                 # Renewable Energy community
        "longitude": lon,
        "latitude": lat,
        "format": "JSON"
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    # Extract the GHI values into a list
    ghi_values = data['properties']['parameter']['ALLSKY_SFC_SW_DWN']
    long_term_monthly = ghi_values['ANN']
    
    # CONVERSION: Multiply by 365 to get the Yearly Total Sum
    annual_total_sum = long_term_monthly * 365
    
    return annual_total_sum


# -- data wrangling
gdf = gdf.to_crs(4326)
# combine all geometries
geom = shapely.unary_union(gdf['geometry'])
# centroid
xy = (geom.centroid.x, geom.centroid.y)
lat, lon = xy[1], xy[0] 

#- execute function
annual_avg = get_ghi_data(lat, lon)
print(f"Annual Average GHI: {round(annual_avg, 2)} kWh/m²/year")

Annual Average GHI: 2029.76 kWh/m²/year


<div class="alert alert-block alert-success"><b>Annual Solar Potential (MWh)</b></div>

We use a simplified formula to provide a clear baseline.

$$
\text{Potential (MWh)} = \frac{(\text{Surface Area} \times \text{utilization factor}) \times \text{GHI}_{\text{annual}} \times 0.2}{1000}
$$


<div class="alert alert-block alert-warning"><b></b>

<sub>***Theoretical Framework:** The annual energy output of a photovoltaic system (E) is determined by the product of the total solar resource (GHI), the active area of the array (A), and the system's overall efficiency (η), adjusted by a Performance Ratio (PR) to account for real-world losses. — based on [NREL (2022)](https://joint-research-centre.ec.europa.eu/photovoltaic-geographical-information-system-pvgis_en) & [IEC 61724-1](https://webstore.iec.ch/en/publication/65561). We then adapt this formula and represents a combined value of 25% nominal panel efficiency and a 0.80 Performance Ratio with a **single 0.20 system efficiency value and account for usable area**, a heuristic for gabled roofs.*</sub> 
</div>

<div class="alert alert-block alert-danger"><b>Your Participation! </b>
    
**Fill in the `utilization_factor` below** 
</div>

As a **'rule-of-thumb'** a community with traditional gabled houses: `utilization_factor = 0.4` *(less than half)*, while a high-density suburb with flat-roofed apartments: `utilization_factor = 0.6`

In [34]:
# on average, how much of the roof faces the sun? Adjust based on roof types
utilization_factor = 0.4 

In [35]:
# Potential (MWh) = (Area * GHI * 0.20) / 1000
blds['solar_mwh'] = ((blds['area'] * utilization_factor) * (annual_avg) * 0.20) / 1000
blds.head(2)

,id,osm_id,building,building:levels,building_height,plus_code,ground_height,roof_height,address,amenity,...,building:use,geometry,pop,area,volume,bvpc,solar_ids,generator:method,has_solar,solar_mwh
7,osm_656840974,656840974,house,1,4.1,4FRWFFM9+X98,170.33,174.43,39 Dove Lane Mamre 7347 Cape Town,NaN,...,NaN,"POLYGON ((264864.982 6288741.008, 264864.617 6...",6.0,38.655779,158.488694,26.414782,[],[],False,6.276972
8,osm_656840975,656840975,house,1,4.1,4FRWFFM9+X9X,169.70,173.80,37 Dove Lane Mamre 7347 Cape Town,NaN,...,NaN,"POLYGON ((264865.35 6288749.141, 264865.026 62...",6.0,39.833617,163.317830,27.219638,[1096772168],[thermal],True,6.468231


In [36]:
# Calculate the average annual solar potential
average_solar_potential = blds['solar_mwh'].mean()

print(" \033[1mThe average solar potential, per home\033[0m , for", jparams['FocusArea'], "is:", round(average_solar_potential,2), "MWh/year")

 The average solar potential, per home , for Mamre is: 17.04 MWh/year


<div style="text-align: left;"> 
<small> <b>NB:</b> this number includes the <a href="https://wiki.openstreetmap.org/wiki/Tag:building%3Dgarage">OpenStreetMap <code>building=garage</code></a> building type. Go to <code>Cell &plusmn;27</code> to exclude this building type from the estimate.</small>
</div>
<br>
<div class="alert alert-block alert-info"><b></b>
   
What does this **MWh/year** value mean?
<br>
    
To put the value in context, **15 MWh/year**:  
- is enough to provide **100% of the electricity** for 4 to 5 average UK homes *(which use ~3.4 MWh each)* or 1.5 average US homes *(~10.7 MWh each)*
- is enough power to drive an Electric Vehicle for **75,000 kilometers** *--that’s almost two full trips around the Earth*.
- saves roughly **10 metric tons of Carbon Dioxide** from entering the atmosphere.
    
</div>

<div class="alert alert-block alert-danger"><b>Sanity Check!</b>
<br>    

- In <code>Cell &plusmn;27</code> we excluded non-residential building types `=office, commercial, retail, warehouse, industrial, etc.` from the analysis.
         
- Cape Town typically yields ~1.6–1.7 MWh **per installed kWp per year**; the higher per-household values reported here reflect **rooftop potential derived from available area** *--that considers a `utilization_factor`*, not a 1 kWp system.

    A 1 kWp solar PV system requires approximately 5–8 m² of panel area (e.g. panels of roughly 1 m × 1.7 m, depending on technology).

    We are NOT asking: How much energy (MWh/year) would a single 1 kWp PV system generate on a roof?  
    We are asking: **How much energy (MWh/year) could these roofs harvest, given their available area and a realistic utilization factor?**

- The **BVPC Warning** applies here too. These are **LoD1 3D City Models**, which represent buildings as simple extrusions.

  LoD2 models *—that capture roof form (e.g. gable, hipped, mansard, domes)—* would provide more representative estimates of both **BVPC and Average Annual Solar Potential**. In such cases, the `utilization_factor` becomes less critical, as usable roof geometry is explicitly modelled. 

</div>

## 3. Interactive Visualization

You might want to create and share an `html` visualization.

<div class="alert alert-block alert-warning"><b> </b>  
    
_In this example we identify building stock by **color** but you are limited only through your imagination and the data you have access too_
</div>

In [37]:
gdf.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

<img src="../data/proj.png" alt="proj" width="550" align="right"/>
<br>
<br>

We need a ***Geographic*** Coordinate Reference System.

        <Geographic 2D CRS: EPSG:4326>
        Name: WGS 84
        Axis Info [ellipsoidal]:
        - Lat[north]: Geodetic latitude (degree)
        - Lon[east]: Geodetic longitude (degree)
        Area of Use:
        - name: World.
        - bounds: (-180.0, -90.0, 180.0, 90.0)
        Datum: World Geodetic System 1984 ensemble
        - Ellipsoid: WGS 84
        - Prime Meridian: Greenwich

In [38]:
#- the pseudo-3Dviz needs geographic coords
gdf = gdf.to_crs(4326)

In [39]:
# -- get the location for 3Dviz

# combine all geometries
geom = shapely.unary_union(gdf['geometry'])
# centroid
xy = (geom.centroid.x, geom.centroid.y)

# bounding box
#minx, miny, maxx, maxy = geom.bounds
#bbox = [minx, miny, maxx, maxy]

In [40]:
# have a look at the building type and amenities available
gdf['building'].unique()

array(['yes', 'church', 'house', 'cabin', 'public', 'civic', 'office',
       'retail', 'clinic', 'school', 'garage', 'greenhouse', 'roof',
       'kindergarten', 'construction', 'clubhouse', 'guest_house',
       'service', 'detached', 'shed'], dtype=object)

<a id='Section2a'></a>
<div class="alert alert-block alert-success"><b>Building Stock:</b> To differentiate a school, housing, retail, healthcare and community focused facilities (library, municipal office, community centre) we color the buildings - we harvest the osm tags [amenity and building type] directly.</div>

In [41]:
# colour buildings based on use / amenity
def color(bld):
    #- formal house
    if bld == 'house' or bld == 'semidetached_house' or bld == 'terrace': #- add maisonette, duplex, etc. 
        return [255, 255, 204]                        #-grey
    if bld == 'apartments':
        return [252, 194, 3]                          #-orange 
    #- informal structure / social housing / student
    if bld == 'residential' or bld == 'dormitory' or bld == 'student' or bld == 'cabin':
        return [119, 3, 252]                          #-purple
        
    if bld == 'garage' or bld == 'parking':
        return [3, 132, 252]                          #-blue        
    if bld == 'retail' or bld == 'supermarket':
        return [253, 141, 60]
    if bld == 'office' or bld == 'commercial':
        return [185, 206, 37]
    if bld == 'school' or bld == 'kindergarten' or bld == 'university' or bld == 'college':
        return [128, 0, 38]
    if bld == 'clinic' or bld == 'doctors' or bld == 'hospital':
        return [89, 182, 178]
    if bld == 'community_centre' or bld == 'service' or bld == 'post_office' or bld == 'hall' \
    or bld ==  'townhall' or bld == 'police' or bld == 'library' or bld == 'fire_station' :
        return [181, 182, 89]
    if bld == 'warehouse' or bld == 'industrial':
        return [193, 255, 193]
    if bld == 'hotel':
        return [139, 117, 0]
    if bld == 'church' or bld == 'mosque' or bld == 'synagogue':
        return [225, 225, 51]
    else:
        return [255, 255, 204]

gdf["fill_color"] = gdf['building'].apply(lambda x: color(x))

In [42]:
#- look
gdf.head(2)

,id,osm_id,building,building:levels,building_height,plus_code,ground_height,roof_height,address,amenity,operator,residential,min_height,bottom_roof_height,building:use,geometry,fill_color
0,osm_328118446,328118446,yes,1,4.1,4FRWFFVC+P79,179.18,183.28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POLYGON ((18.47060329825786 -33.50578760181099...,"[255, 255, 204]"
1,osm_328118447,328118447,church,2,6.9,4FRWFFVC+H9Q,178.30,185.20,Moravian Church South Africa Kerk Street Mamre...,place_of_worship,NaN,NaN,NaN,NaN,NaN,POLYGON ((18.470915303324528 -33.5059178036112...,"[225, 225, 51]"


In [43]:
#- create pseudo-3D viz

file = '../result/interactiveAlt.html' # will name and save html here

html = city3D.create_maplibre_3Dviz(
    result_dir = file, 
    buildings_gdf = gdf,
    center = xy
)

city3D.show_interactive_html(html)

**on a laptop without a mouse:**

- `trackpad left-click drag-left` and `-right`;
- `Ctrl left-click drag-up`, `-down`, `-left` and `-right` to rotate and so-on and
- `+` next to Backspace zoom-in and `-` next to `+` zoom-out.

**Now you do your community.** ~ If your area needs [OpenStreetMap](https://en.wikipedia.org/wiki/OpenStreetMap)  data and you want to contribute please follow the [Guide](https://wiki.openstreetmap.org/wiki/Beginners%27_guide).

<a id='Section3'></a>

<div class="alert alert-block alert-warning"><b>  </b>  
    
**To understand the performance in an [Urban setting](https://en.wikipedia.org/wiki/Urban_area) change `cell [2]` above:**

<span style="color:black">**jparams**</span> = <span style="color:Darkred">uEstate</span> <span style="color:black">(['University Estate'](https://en.wikipedia.org/wiki/University_Estate)) or</span> <span style="color:Darkred">sRiver</span>  <span style="color:black">(['Salt River'](https://en.wikipedia.org/wiki/Salt_River,_Cape_Town)) or</span> <span style="color:Darkred">obs</span> <span style="color:black">(['Observatory'](https://en.wikipedia.org/wiki/Observatory,_Cape_Town)) *(with residents per formal house = 4 | 5 in Salt River and residents per informal structure = 3)*</span> <span style="color:black">and</span>  <span style="color:Darkred">cput</span> <span style="color:black">('Cape Peninsula University of Technology (Bellville Campus)')</span>       
</div>

## 4. Possible Secondary and Tertiary level *conversations starters*:

<div class="alert alert-block alert-success"><b>communicate and exchange ideas and understanding</b></div>

| **Topic**                                | **Secondary Level Questions**                                                                                                                                                                                   | **Tertiary Level Questions**                                                                                                                                                                                                                   |
|------------------------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **Basic Understanding and Observations** | - What types of buildings are most common in the area (houses, apartments, retail, etc.)?<br>- Can you identify any patterns in the distribution of different types of buildings (e.g., are retail stores concentrated in certain areas)? | - How does the building stock composition (e.g., ratio of houses) correlate with the population? *demographics (e.g., age distribution, household size) for the area will strengthen the analysis!* <br>- Analyze the relationship between building density and population. What urban planning theories can explain this relationship? |
| **Spatial Relationships and Impacts**    | - How does the location of residential areas compare to the location of retail and commercial areas?<br>- What impact might the density and distribution of buildings have on local traffic and transportation?<br>- How might the population distribution affect the demand for local services such as schools, hospitals, and parks? | - Evaluate the accessibility of essential services (e.g., healthcare, education) in relation to the population and building types.<br>- Assess the potential social and economic impacts of a proposed new residential or commercial development in the area.                  |
| **Socioeconomic and Environmental Considerations** | - Are there any correlations between the types of housing available and the household size? *additional demographics (e.g., income level) for the area will strengthen the analysis!*<br>- How might the current building stock and population influence the local economy? *demographics (e.g., age distribution, household size) for the area will strengthen the analysis!*<br>- What are some potential environmental impacts of the current building distribution, such as green space availability or pollution levels? | - How does the current building stock support or hinder sustainable development goals (e.g., energy efficiency, reduced carbon footprint)?<br>- What strategies could be implemented to increase the resilience of the community to environmental or economic changes?                       |
| **Future Planning and Development**      | - Based on the current building stock and population metrics, what areas might benefit from additional housing or commercial development?<br>- How could urban planners use this information to improve the quality of life in the area?<br>- What changes would you recommend to better balance residential, commercial, and recreational spaces? | - How might different zoning regulations impact the distribution of residential, commercial, and industrial buildings in the future?<br>- Propose urban design solutions that could improve the sustainability and livability of the area, considering both current metrics and future projections. |
| **Quantitative and Qualitative Research** | |- Design a research study to investigate the impact of building type diversity on community wellbeing. What methodologies would you use?<br>- Analyze historical data to understand trends in building development and population growth. How have these trends shaped the current urban landscape?<br>- Conduct a SWOT analysis (Strengths, Weaknesses, Opportunities, Threats) of the area based on the building stock and population metrics. |

***

**Now you do your community.** ~ If your area needs [OpenStreetMap](https://en.wikipedia.org/wiki/OpenStreetMap) data and you want to contribute please follow the [Guide](https://wiki.openstreetmap.org/wiki/Beginners%27_guide). 